# Gaussian and point-source simulation

[Colab Link](https://colab.research.google.com/github/casangi/astroviper/blob/main/docs/distributed_applications_tutorials/simulation/gaussian_component_simulation.ipynb)

Simulate a sky of **two point sources and two elliptical Gaussian sources**, image it with the
**Adaptive Scale Pixel (ASP) CLEAN** deconvolver and verify that the input fluxes are recovered
after primary-beam correction.

A Gaussian source is simulated as a point source whose visibilities are multiplied by the
analytic Fourier transform of its sky Gaussian.  That transform is the imaging **restore**
module's ``elliptical_gaussian_uv_taper`` -- the same parametrisation (FWHM major / minor axes
and position angle) as the clean beam that the restore step convolves into the model, so the
simulator and the imager share one Gaussian definition and one implementation.

---
## API

In [ ]:
from astroviper.distributed_applications.simulation import simulate_processing_set

simulate_processing_set?

## Install AstroVIPER

In [ ]:
import os
from importlib.metadata import version

try:
    import astroviper  # noqa: F401

    print("Using astroviper version", version("astroviper"))
except ImportError:
    os.system("pip install --upgrade astroviper")
    import astroviper  # noqa: F401

    print("Installed astroviper version", version("astroviper"))

In [ ]:
# Set True for interactive (zoom / pan) plots via the ipympl widget backend
# (``pip install ipympl``).  Keep False for the automated notebook tests, which
# execute headless.
INTERACTIVE_PLOTS = False

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from astropy.coordinates import SkyCoord
from IPython import get_ipython

get_ipython().run_line_magic("matplotlib", "widget" if INTERACTIVE_PLOTS else "inline")

xr.set_options(display_style="html")
ARCSEC_TO_RAD = np.pi / (180 * 3600)

In [ ]:
from toolviper.dask.client import local_client

viper_client = local_client(cores=4, memory_limit="4GB")
viper_client

## ALMA 12 m array

Thirty 12 m dishes of the full ALMA layout.

In [ ]:
from astroviper.utils.telescope_layout import read_telescope_layout

alma_all = read_telescope_layout("alma.all")
is_12m = alma_all.ANTENNA_DISH_DIAMETER.values == 12.0
antenna_xds = alma_all.isel(antenna_name=np.where(is_12m)[0][:30])
n_antenna = antenna_xds.sizes["antenna_name"]
print(n_antenna, "antennas")

## The sky: two point sources and two Gaussians

Sources are placed by image pixel (``sin_pixel_to_celestial_coord``).  Each Gaussian is
described by its **integrated** flux and its FWHM shape ``[major, minor, position angle]``;
the position angle follows the imaging clean-beam convention.

In [ ]:
from astroviper.utils.coordinate_transforms import sin_pixel_to_celestial_coord

image_size = np.array([256, 256])
cell_size_arcsec = 0.5
cell_size = np.array([-cell_size_arcsec, cell_size_arcsec]) * ARCSEC_TO_RAD

phase_center = SkyCoord(ra="19h59m28.5s", dec="-40d44m01.5s", frame="icrs")
phase_center_ra_dec = np.array([phase_center.ra.rad, phase_center.dec.rad])[None, :]

pixels_point = np.array([[128, 128], [88, 168]])
flux_point = [1.0, 1.5]  # Jy
pixels_gaussian = np.array([[168, 168], [128, 68]])
flux_gaussian = [2.0, 1.2]  # Jy (integrated)
shape_gaussian = np.array(
    [
        [4.0 * ARCSEC_TO_RAD, 4.0 * ARCSEC_TO_RAD, 0.0],  # round, 4'' FWHM
        [6.0 * ARCSEC_TO_RAD, 2.5 * ARCSEC_TO_RAD, np.deg2rad(30)],  # elongated
    ]
)

point_source_ra_dec = sin_pixel_to_celestial_coord(
    phase_center_ra_dec[0], image_size, cell_size, pixels_point
)[None, :, :]
gaussian_source_ra_dec = sin_pixel_to_celestial_coord(
    phase_center_ra_dec[0], image_size, cell_size, pixels_gaussian
)[None, :, :]
point_source_flux = np.stack([np.array([f, 0, 0, f]) for f in flux_point])[
    :, None, None, :
]
gaussian_source_flux = np.stack([np.array([f, 0, 0, f]) for f in flux_gaussian])[
    :, None, None, :
]

## Simulate

In [ ]:
from astroviper.utils.beam_models import airy_disk_model

result = simulate_processing_set(
    ps_store="gaussian_sim.ps.zarr",
    antenna_xds=antenna_xds,
    time_params={
        "time_start": "2019-10-03T19:00:00.000",
        "time_delta": 1600.0,
        "n_samples": 6,
    },
    frequency_params={
        "freq_start": 90e9,
        "freq_delta": 0.5e9,
        "n_channels": 2,
        "channel_width": 0.5e9,
        "spectral_window_name": "Band3",
    },
    polarization=["XX", "YY"],
    point_source_flux=point_source_flux,
    point_source_ra_dec=point_source_ra_dec,
    gaussian_source_flux=gaussian_source_flux,
    gaussian_source_ra_dec=gaussian_source_ra_dec,
    gaussian_source_shape=shape_gaussian,
    phase_center_ra_dec=phase_center_ra_dec,
    beam_models=[airy_disk_model("alma")],
    beam_model_map=np.zeros(n_antenna, dtype=int),
    n_time_chunks=2,
    n_frequency_chunks=2,
    overwrite=True,
)
result["timing_node_tasks"][["task_id", "T_uvw", "T_visibilities", "T_write"]]

## Validate the processing set against the MSv4 schema

``xradio.schema.check.check_datatree`` checks every dataset of the processing set against the
MSv4 schema.

In [ ]:
from xradio.measurement_set import open_processing_set
from xradio.schema.check import check_datatree

ps_xdt = open_processing_set("gaussian_sim.ps.zarr")
issues = check_datatree(ps_xdt)
print(issues)
assert str(issues) == "No schema issues found"

## Image with the Adaptive Scale Pixel (ASP) deconvolver

ASP CLEAN models the sky as Gaussians of adaptive scales, a good match for this field.  The
restored sky is primary-beam corrected (``primary_beam_correction=True``, CASA ``pbcor``).

In [ ]:
from xradio.image import load_image, make_empty_sky_image

from astroviper.distributed_applications.imaging import image_cube_single_field


def image_simulation(ps_store, image_store, niter=3000, deconvolver="asp"):
    """Image a simulated processing set with AstroVIPER (natural weighting)."""
    ps_xdt = open_processing_set(ps_store)
    combined = ps_xdt.xr_ps.get_combined_field_and_source_xds()
    phase_direction = combined.FIELD_PHASE_CENTER_DIRECTION.sel(
        field_name=combined.attrs["center_field_name"]
    ).values
    image_cube_single_field(
        ps_store=ps_store,
        image_store=image_store,
        image_params={
            "image_size": list(image_size),
            "cell_size": cell_size,
            "phase_direction": phase_direction,
            "frequency_coords": ps_xdt.xr_ps.get_freq_axis().values,
            "polarization_coords": ["I"],
            "time_coords": [0],
            "fft_padding": 1.2,
            "cpp_gridder": True,
        },
        imaging_weights_params={
            "weighting": "natural",
            "robust": 0.5,
            "casa_weighting_implementation": True,
        },
        iteration_control_params={
            "niter": niter,
            "nmajor": -1,
            "threshold": 2e-4,
            "gain": 0.1,
            "cyclefactor": 1.5,
            "cycleniter": -1,
            "minpsffraction": 0.05,
            "maxpsffraction": 0.8,
            "primary_beam_limit": 0.2,
        },
        gridder="prolate_spheroidal",
        deconvolver=deconvolver,
        scan_intents="OBSERVE_TARGET#ON_SOURCE",
        image_data_variables_keep=[
            "sky_residual",
            "point_spread_function",
            "primary_beam",
            "beam_fit_params_point_spread_function",
            "sky_model",
            "mask",
        ],
        processing_set_data_group_name="base",
        single_precision_image=False,
        processing_function_threads=1,
        n_chunks=2,
        overwrite=True,
        restore=True,
        primary_beam_correction=True,
    )
    return add_sky_coordinates(load_image(image_store))


def add_sky_coordinates(img_xds):
    """Attach 2-D right_ascension / declination pixel coordinates (SIN grid)."""
    img_xds.attrs["type"] = "image_dataset"  # the xr_img accessor checks this
    reference_direction = img_xds.attrs["coordinate_system_info"][
        "reference_direction"
    ]["data"]
    template = make_empty_sky_image(
        phase_center=np.asarray(reference_direction),
        image_size=[img_xds.sizes["l"], img_xds.sizes["m"]],
        cell_size=img_xds.xr_img.get_lm_cell_size(),
        frequency_coords=img_xds.frequency.values,
        pol_coords=list(img_xds.polarization.values),
        time_coords=list(img_xds.time.values),
        do_sky_coords=True,
    )
    # assign by raw values: aligning on the template's float l / m coords
    # would reindex the sky coordinates to NaN
    return img_xds.assign_coords(
        right_ascension=(("l", "m"), template.right_ascension.values),
        declination=(("l", "m"), template.declination.values),
    )

In [ ]:
img_xds = image_simulation("gaussian_sim.ps.zarr", "gaussian_sim.img.zarr")
print("clean beam [arcsec, arcsec, deg]:")
beam_fit = img_xds.BEAM_FIT_PARAMS_POINT_SPREAD_FUNCTION.isel(
    time=0, frequency=0, polarization=0
).values
print(
    f"  {beam_fit[0] / ARCSEC_TO_RAD:.2f} x {beam_fit[1] / ARCSEC_TO_RAD:.2f}, "
    f"pa {np.rad2deg(beam_fit[2]):.1f}"
)

## All imaging data products

The headline view: every image-plane product in one figure, with image **pixels** on the
bottom / left axes and **right ascension / declination** on the top / right.  The two
Gaussians are visibly extended in ``SKY_RESTORED`` while the ASP model
(``SKY_MODEL``) concentrates their flux.

In [ ]:
colorbar_labels = {
    "SKY_RESTORED": "Jy/beam",
    "SKY_RESTORED_PRIMARY_BEAM_CORRECTED": "Jy/beam",
    "SKY_RESIDUAL": "Jy/beam",
    "SKY_MODEL": "Jy/pixel",
    "POINT_SPREAD_FUNCTION": "response",
    "PRIMARY_BEAM": "power response",
    "MASK": "boolean",
}


def _sky_coordinate_axes(ax, img_xds):
    from astropy import units as u
    from astropy.coordinates import Angle

    ra = img_xds.right_ascension.values
    dec = img_xds.declination.values
    n_l, n_m = ra.shape
    top = ax.secondary_xaxis("top")
    top.set_xticks(np.linspace(0, n_l - 1, 3))
    top.set_xticklabels(
        [
            Angle(ra[int(pix), n_m // 2], u.rad).to_string(
                unit=u.hourangle, precision=1
            )
            for pix in np.linspace(0, n_l - 1, 3)
        ],
        fontsize=7,
    )
    top.set_xlabel("right ascension", fontsize=8)
    right = ax.secondary_yaxis("right")
    right.set_yticks(np.linspace(0, n_m - 1, 3))
    right.set_yticklabels(
        [
            Angle(dec[n_l // 2, int(pix)], u.rad).to_string(unit=u.deg, precision=0)
            for pix in np.linspace(0, n_m - 1, 3)
        ],
        fontsize=7,
    )
    right.set_ylabel("declination", fontsize=8)


def show_all_products(img_xds, suptitle, frequency=0, polarization=0, n_cols=4):
    """One-figure montage of every image-plane data product."""
    variables = [
        name for name in img_xds.data_vars if {"l", "m"} <= set(img_xds[name].dims)
    ]
    n_rows = -(-len(variables) // n_cols)
    fig, axes = plt.subplots(
        n_rows, n_cols, figsize=(4.6 * n_cols, 4.3 * n_rows), constrained_layout=True
    )
    for ax in np.ravel(axes):
        ax.set_axis_off()
    for ax, variable in zip(np.ravel(axes), variables, strict=False):
        ax.set_axis_on()
        plane = (
            img_xds[variable]
            .isel(time=0, frequency=frequency, polarization=polarization)
            .values
        )
        im = ax.imshow(plane.T, origin="lower", cmap="viridis")
        ax.set_title(variable, fontsize=10)
        ax.set_xlabel("l [pixel]")
        ax.set_ylabel("m [pixel]")
        _sky_coordinate_axes(ax, img_xds)
        fig.colorbar(im, ax=ax, shrink=0.75, label=colorbar_labels.get(variable, ""))
    fig.suptitle(
        f"{suptitle} (channel {frequency}, "
        f"{img_xds.frequency.values[frequency] / 1e9:.2f} GHz)"
    )
    return fig


show_all_products(img_xds, "points + Gaussians, ASP CLEAN")
plt.show()

## Flux recovery after primary-beam correction

For each source, read the primary-beam-corrected restored image
(``SKY_RESTORED_PRIMARY_BEAM_CORRECTED``):

* **point sources** -- the corrected peak equals the sky flux;
* **Gaussians** -- the corrected image is the sky Gaussian convolved with the clean beam,
  so the **aperture-integrated** flux (sum x pixel area / beam area) equals the integrated
  sky flux, and the peak equals the analytic convolution
  ``flux x sqrt(det(C_beam) / det(C_beam + C_source))`` (Gaussian covariances add under
  convolution).

In [ ]:
import pandas as pd


def covariance(major, minor, pa):
    """Sky covariance matrix of a Gaussian from FWHM axes and position angle."""
    sigma_major = (major / (2 * np.sqrt(2 * np.log(2)))) ** 2
    sigma_minor = (minor / (2 * np.sqrt(2 * np.log(2)))) ** 2
    e = np.array([np.sin(pa), np.cos(pa)])
    p = np.array([np.cos(pa), -np.sin(pa)])
    return sigma_major * np.outer(e, e) + sigma_minor * np.outer(p, p)


corrected = img_xds.SKY_RESTORED_PRIMARY_BEAM_CORRECTED.isel(
    time=0, frequency=0, polarization=0
).values
primary_beam = img_xds.PRIMARY_BEAM.isel(time=0, frequency=0, polarization=0).values
beam_area = np.pi / (4 * np.log(2)) * beam_fit[0] * beam_fit[1]
pixel_area = float(np.abs(cell_size[0] * cell_size[1]))
beam_covariance = covariance(*beam_fit)

rows = []
for pix, flux in zip(pixels_point, flux_point, strict=True):
    rows.append(
        {
            "source": f"point {pix}",
            "input flux [Jy]": flux,
            "primary beam": primary_beam[pix[0], pix[1]],
            "corrected peak [Jy/beam]": corrected[pix[0], pix[1]],
            "expected peak [Jy/beam]": flux,
            "integrated flux [Jy]": np.nan,
        }
    )
for pix, flux, shape in zip(
    pixels_gaussian, flux_gaussian, shape_gaussian, strict=True
):
    box = corrected[pix[0] - 25 : pix[0] + 25, pix[1] - 25 : pix[1] + 25]
    integrated = np.nansum(box) * pixel_area / beam_area
    expected_peak = flux * np.sqrt(
        np.linalg.det(beam_covariance)
        / np.linalg.det(beam_covariance + covariance(*shape))
    )
    rows.append(
        {
            "source": f"gaussian {pix}",
            "input flux [Jy]": flux,
            "primary beam": primary_beam[pix[0], pix[1]],
            "corrected peak [Jy/beam]": corrected[pix[0], pix[1]],
            "expected peak [Jy/beam]": expected_peak,
            "integrated flux [Jy]": integrated,
        }
    )
table = pd.DataFrame(rows).round(4)
table

In [ ]:
# the same checks the component test makes: fluxes are recovered
for row in rows[:2]:
    np.testing.assert_allclose(
        row["corrected peak [Jy/beam]"], row["input flux [Jy]"], rtol=0.02
    )
for row in rows[2:]:
    np.testing.assert_allclose(
        row["integrated flux [Jy]"], row["input flux [Jy]"], rtol=0.05
    )
    np.testing.assert_allclose(
        row["corrected peak [Jy/beam]"], row["expected peak [Jy/beam]"], rtol=0.03
    )
print("all fluxes recovered")

## Clean up

In [ ]:
import shutil

for path in ["gaussian_sim.ps.zarr", "gaussian_sim.img.zarr"]:
    shutil.rmtree(path, ignore_errors=True)
viper_client.close()